# Shor's Algorithm (Simplified)

Shor's algorithm factors integers in polynomial time by reducing factoring to **order finding**: given $a$ and $N$, find the smallest $r$ such that $a^r \equiv 1 \pmod{N}$.

The quantum subroutine uses:
1. A superposition over all exponents $x$
2. An oracle computing $a^x \bmod N$ in superposition
3. Inverse QFT to extract the period $r$
4. Classical continued fractions to recover $r$

We demonstrate with $N = 15 = 3 \times 5$.

In [ ]:
import cudaq
import numpy as np
from math import gcd


@cudaq.kernel
def order_finding_a2():
    """Quantum circuit for finding the order of a=2 mod N=15."""
    qubits = cudaq.qvector(8)
    h(qubits[0])
    h(qubits[1])
    h(qubits[2])
    h(qubits[3])
    x(qubits[4])
    cx(qubits[0], qubits[5])
    cx(qubits[0], qubits[6])
    cx(qubits[1], qubits[6])
    cx(qubits[1], qubits[7])
    cx(qubits[2], qubits[5])
    cx(qubits[2], qubits[6])
    cx(qubits[2], qubits[7])
    cx(qubits[3], qubits[5])
    cx(qubits[3], qubits[7])
    swap(qubits[0], qubits[3])
    swap(qubits[1], qubits[2])
    h(qubits[0])
    crz(qubits[0], qubits[1], -np.pi / 2)
    h(qubits[1])
    crz(qubits[0], qubits[2], -np.pi / 4)
    crz(qubits[1], qubits[2], -np.pi / 2)
    h(qubits[2])
    crz(qubits[0], qubits[3], -np.pi / 8)
    crz(qubits[1], qubits[3], -np.pi / 4)
    crz(qubits[2], qubits[3], -np.pi / 2)
    h(qubits[3])

In [ ]:
N = 15
a = 2
print(f"Factoring N = {N}, a = {a}")
print(f"gcd({a}, {N}) = {gcd(a, N)}")

result = cudaq.sample(order_finding_a2, shots_count=1000)
counts = {k: v for k, v in result.items()}
print(f"\nCount register measurements (top 5):")
sorted_counts = sorted(counts.items(), key=lambda x: -x[1])[:5]
for bitstring, count in sorted_counts:
    count_bits = bitstring[:4]
    s = int(count_bits, 2)
    print(f"  |{bitstring}>: {count}  (count = {count_bits} = {s})")

max_count = max(counts.values())
most_common = [k for k, v in counts.items() if v == max_count][0]
s = int(most_common[:4], 2)
print(f"\nMost likely s = {s}")
print(f"s/2^4 = {s}/16 = {s/16:.4f}")

In [ ]:
def continued_fraction(s, denom):
    """Find the denominator r from s/2^n using continued fractions."""
    n = 16
    if s == 0:
        return denom
    frac = s / (2 ** n)
    for r in range(1, denom + 1):
        if abs(frac - round(frac * r) / r) < 0.01:
            return r
    return denom

r = continued_fraction(s, N)
print(f"Continued fraction => period r = {r}")

if r % 2 == 0:
    x1 = pow(a, r // 2, N)
    f1 = gcd(x1 - 1, N)
    f2 = gcd(x1 + 1, N)
    print(f"a^(r/2) mod N = {a}^{r//2} mod {N} = {x1}")
    print(f"gcd({x1 - 1}, {N}) = {f1}")
    print(f"gcd({x1 + 1}, {N}) = {f2}")
    if f1 != 1 and f1 != N:
        print(f"\nFound: {N} = {f1} * {N // f1}")
    elif f2 != 1 and f2 != N:
        print(f"\nFound: {N} = {f2} * {N // f2}")
else:
    print("r is odd, retry with a different a")